# 2025 Statcast pull (raw)

This notebook documents **step 1** of the pipeline: download regular-season pitch-level Statcast data and save it as parquet.

**In scope:** raw Savant columns, regular-season filter, dtype cleanup, dedup keys, optional player name lookup table.

**Out of scope:** feature engineering, pitch-type filtering, swing/whiff labels, train/test splits.

## 1. Why monthly chunks?

Baseball Savant via `pybaseball.statcast()` is slow and can time out on a full season. We pull **one calendar month at a time**, cache each month under `data/raw_parquet/`, then merge into a single season file.

If a month is already cached and the schema version matches, we skip the API call.

## 2. Column contract (schema v3)

All columns are **as returned by Statcast** (no derived fields). See `src/statcast_schema.py` for the list.

| Group | Columns |
|-------|---------|
| Keys | `game_date`, `game_pk`, `at_bat_number`, `pitch_number` |
| Players | `batter`, `pitcher` (MLBAM IDs) |
| Location | `plate_x`, `plate_z`, `sz_top`, `sz_bot` |
| Context | `balls`, `strikes`, `zone`, `on_1b`, `on_2b`, `on_3b`, `outs_when_up`, `p_throws`, `stand`, `pitch_name` |
| Physics / release | `release_speed`, `effective_speed`, `pfx_x`, `pfx_z`, `release_spin_rate`, `spin_axis`, `release_extension`, **`release_pos_x`**, **`release_pos_y`**, **`release_pos_z`** |
| Outcomes | `pitch_type`, `description`, `events`, `estimated_woba_using_speedangle` |

**Required for downstream modeling:** release point (`release_pos_x/y/z`), extension, spin, movement, platoon (`p_throws`, `stand`). The pull **raises an error** if Savant omits any required column.

**All pitch types** are kept at pull time. Competitive-pitch filtering happens in `02_preprocessing.ipynb`.

**Re-pull after schema changes:** bump `SCHEMA_VERSION` in `statcast_schema.py`, then run with `--force`:

```powershell
.\.venv\Scripts\python.exe -m src.statcast_pull --force
```

## 3. Regular season filter

After each API pull we keep rows where:

1. `game_type == 'R'` when that column exists
2. `game_date` is between **2025-03-27** and **2025-09-28**

Spring training and exhibitions are dropped.

## 4. Player lookup table - Images

`batter` and `pitcher` are numeric MLBAM IDs. After the season file is built, we call pybaseball's name lookup (backed by the **Chadwick register** — Baseball's crosswalk of ID systems) and write `data/players.parquet` with `mlbam_id` and `player_name`.

This **does not remove any pitch rows**. It is only for joining names in EDA and the app. The old v1 pipeline *dropped* pitches when a batter ID failed lookup; we are not doing that here.

In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)

try:
    import pandas as pd
    import pyarrow  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing packages in this kernel. In a terminal run:\n"
        "  .venv\\Scripts\\python.exe -m pip install -r requirements.txt\n"
        "Then restart the kernel and select the .venv interpreter."
    ) from exc

print("pandas:", pd.__version__)

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Expected src/ under project root; cwd={NB_DIR}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.statcast_schema import SEASON_START, SEASON_END, STATCAST_KEEP_COLS, regular_season_month_ranges

print(f"Project root: {ROOT}")
print(f"Season: {SEASON_START} → {SEASON_END}")
print(f"Columns ({len(STATCAST_KEEP_COLS)}): {STATCAST_KEEP_COLS}")
print("Monthly pull windows:")
for start, end in regular_season_month_ranges():
    print(f"  {start} → {end}")
print("Setup OK — run the next cell to pull or load existing data.")

Python: c:\Users\tabshire\Desktop\Portfolio\.venv\Scripts\python.exe
pandas: 3.0.3
Project root: C:\Users\tabshire\Desktop\Portfolio
Season: 2025-03-27 → 2025-09-28
Columns (36): ['game_date', 'game_pk', 'at_bat_number', 'pitch_number', 'batter', 'pitcher', 'plate_x', 'plate_z', 'sz_top', 'sz_bot', 'balls', 'strikes', 'zone', 'on_1b', 'on_2b', 'on_3b', 'outs_when_up', 'p_throws', 'stand', 'pitch_name', 'release_speed', 'effective_speed', 'pfx_x', 'pfx_z', 'release_spin_rate', 'spin_axis', 'release_extension', 'release_pos_x', 'release_pos_y', 'release_pos_z', 'pitch_type', 'description', 'events', 'estimated_woba_using_speedangle', 'bat_speed', 'attack_angle']
Monthly pull windows:
  2025-03-27 → 2025-03-31
  2025-04-01 → 2025-04-30
  2025-05-01 → 2025-05-31
  2025-06-01 → 2025-06-30
  2025-07-01 → 2025-07-31
  2025-08-01 → 2025-08-31
  2025-09-01 → 2025-09-28
Setup OK — run the next cell to pull or load existing data.


## 5. Run the pull

First run can take **10–20+ minutes** depending on Savant response times. Re-runs use cached monthly parquet files when schema v3 matches.

**After a schema upgrade** (new release columns), force a full re-download:

```powershell
.\.venv\Scripts\python.exe -m src.statcast_pull --force
```

Or set `FORCE_PULL = True` in the cell below.

In [2]:
import pandas as pd
from pybaseball import cache

from src.statcast_schema import SCHEMA_VERSION
from src.statcast_pull import OUTPUT_FILE, PLAYERS_FILE, build_season_file, pull_summary, season_file_is_current

FORCE_PULL = True  # set True once after schema v3 upgrade, then False for normal runs

if not FORCE_PULL and season_file_is_current(OUTPUT_FILE):
    print(f"Found current season file — loading {OUTPUT_FILE.name} (no API pull).", flush=True)
    season_df = pd.read_parquet(OUTPUT_FILE)
else:
    if OUTPUT_FILE.exists() and not FORCE_PULL:
        print(f"Season file is stale (schema v{SCHEMA_VERSION}) — re-pulling.", flush=True)
    elif FORCE_PULL:
        print(f"FORCE_PULL=True — re-downloading all months (schema v{SCHEMA_VERSION}).", flush=True)
    cache.enable()
    print("Starting pull (prints each month as it runs)...", flush=True)
    season_df = build_season_file(build_players=True, force=FORCE_PULL)

summary = pull_summary(season_df)
summary

FORCE_PULL=True — re-downloading all months (schema v4).
Starting pull (prints each month as it runs)...
Cleared 14 cached monthly file(s) (--force).
Pulling 2025-03-27 to 2025-03-31 ...
This is a large query, it may take a moment to complete


100%|██████████| 5/5 [00:01<00:00,  2.51it/s]


Saved 19,155 regular-season rows to statcast_2025-03-27_to_2025-03-31.parquet
Pulling 2025-04-01 to 2025-04-30 ...
This is a large query, it may take a moment to complete


100%|██████████| 30/30 [00:05<00:00,  5.30it/s]


Saved 114,767 regular-season rows to statcast_2025-04-01_to_2025-04-30.parquet
Pulling 2025-05-01 to 2025-05-31 ...
This is a large query, it may take a moment to complete


...

100%|██████████| 31/31 [00:11<00:00,  2.64it/s]


Saved 124,298 regular-season rows to statcast_2025-08-01_to_2025-08-31.parquet
Pulling 2025-09-01 to 2025-09-28 ...
This is a large query, it may take a moment to complete


100%|██████████| 28/28 [00:05<00:00,  5.52it/s]


Saved 110,338 regular-season rows to statcast_2025-09-01_to_2025-09-28.parquet
Looking up 1,469 unique batter/pitcher IDs...
  Player lookup batch 1/3 (500 IDs)...
  Player lookup batch 2/3 (500 IDs)...
  Player lookup batch 3/3 (469 IDs)...
Saved player lookup: C:\Users\tabshire\Desktop\Portfolio\data\players.parquet (1,469 IDs)


{'rows': 711897,
 'columns': ['game_date',
  'game_pk',
  'at_bat_number',
  'pitch_number',
  'batter',
  'pitcher',
  'plate_x',
  'plate_z',
  'sz_top',
  'sz_bot',
  'balls',
  'strikes',
  'zone',
  'on_1b',
  'on_2b',
  'on_3b',
  'outs_when_up',
  'p_throws',
  'stand',
  'pitch_name',
  'release_speed',
  'effective_speed',
  'pfx_x',
  'pfx_z',
  'release_spin_rate',
  'spin_axis',
  'release_extension',
  'release_pos_x',
  'release_pos_y',
  'release_pos_z',
  'pitch_type',
  'description',
  'events',
  'estimated_woba_using_speedangle',
  'bat_speed',
  'attack_angle'],
 'schema_version': 4,
 'date_min': '2025-03-27 00:00:00',
 'date_max': '2025-09-28 00:00:00',
 'unique_batters': 673,
 'unique_pitchers': 873,
 'release_null_pct': {'release_pos_x': 0.4,
  'release_pos_y': 0.4,
  'release_pos_z': 0.4,
  'release_extension': 0.4,
  'release_spin_rate': 0.6,
  'p_throws': 0.0,
  'stand': 0.0},
 'null_pct': {'game_date': 0.0,
  'game_pk': 0.0,
  'at_bat_number': 0.0,
  'pitch_

## 6. Sanity checks

Next: **Batter/Pitcher Name Check** (last section — scroll to the bottom of this notebook).

In [3]:
import pandas as pd

from src.statcast_pull import OUTPUT_FILE, PLAYERS_FILE, pull_summary

if "season_df" not in globals():
    season_df = pd.read_parquet(OUTPUT_FILE)
if "summary" not in globals():
    summary = pull_summary(season_df)

print(f"Season file: {OUTPUT_FILE}")
print(f"Players file: {PLAYERS_FILE}")
print(f"Rows: {len(season_df):,}")
print(f"Dates: {season_df['game_date'].min()} → {season_df['game_date'].max()}")
print(f"Batters: {season_df['batter'].nunique():,} | Pitchers: {season_df['pitcher'].nunique():,}")
print(f"Pitch types: {season_df['pitch_type'].nunique()} unique codes")

players = pd.read_parquet(PLAYERS_FILE)
print(f"Player lookup rows: {len(players):,}")

display(season_df.head(3))
display(season_df[["pitch_type", "description"]].value_counts().head(15))
display(pd.Series(summary["null_pct"]).sort_values(ascending=False).head(10))
display(players.head(5))

Season file: C:\Users\tabshire\Desktop\Portfolio\data\statcast_2025.parquet
Players file: C:\Users\tabshire\Desktop\Portfolio\data\players.parquet
Rows: 711,897
Dates: 2025-03-27 00:00:00 → 2025-09-28 00:00:00


Batters: 673 | Pitchers: 873
Pitch types: 18 unique codes
Player lookup rows: 1,469


,game_date,game_pk,at_bat_number,pitch_number,batter,pitcher,plate_x,plate_z,sz_top,sz_bot,...,release_extension,release_pos_x,release_pos_y,release_pos_z,pitch_type,description,events,estimated_woba_using_speedangle,bat_speed,attack_angle
0,2025-03-27,778545,1,1,595777,650633,-0.524909,2.528382,3.279838,1.580001,...,5.9,-3.10,54.599998,5.81,FF,called_strike,<NA>,NaN,NaN,NaN
1,2025-03-27,778545,1,2,595777,650633,0.309268,2.040064,3.355339,1.706879,...,5.7,-3.27,54.759998,5.45,SI,ball,<NA>,NaN,NaN,NaN
2,2025-03-27,778545,1,3,595777,650633,-0.285129,1.713766,3.290000,1.580000,...,5.9,-3.25,54.590000,5.40,CH,hit_into_play,single,0.185,78.599998,10.611278


pitch_type  description    
FF          ball               75839
            foul               48825
            called_strike      38853
            hit_into_play      37192
SI          ball               34207
SL          ball               33018
CH          ball               26765
SI          called_strike      24587
            hit_into_play      23252
FF          swinging_strike    20390
SI          foul               19563
ST          ball               19100
FC          ball               17574
SL          hit_into_play      17463
CU          ball               16690
Name: count, dtype: int64

on_3b                              90.5
on_2b                              81.0
estimated_woba_using_speedangle    74.6
events                             74.3
on_1b                              69.5
bat_speed                          53.8
attack_angle                       53.8
spin_axis                           0.6
release_spin_rate                   0.6
release_extension                   0.4
dtype: float64

,mlbam_id,player_name
0,621345,A. J. Minter
1,640462,A. J. Puk
2,676879,Aaron Ashby
3,607481,Aaron Bummer
4,650644,Aaron Civale


## Batter/Pitcher Name Check

Pitch rows store **`batter`** and **`pitcher`** as MLBAM numeric IDs only. Human-readable names live in **`data/players.parquet`**, built during the pull.

Join on `players.mlbam_id` twice — once for the batter, once for the pitcher — to add `batter_name` and `pitcher_name`.

In [4]:
import pandas as pd

from src.statcast_pull import OUTPUT_FILE, PLAYERS_FILE

if "season_df" not in globals():
    season_df = pd.read_parquet(OUTPUT_FILE)

players = pd.read_parquet(PLAYERS_FILE)

named = (
    season_df.merge(
        players.rename(columns={"mlbam_id": "batter", "player_name": "batter_name"}),
        on="batter",
        how="left",
    )
    .merge(
        players.rename(columns={"mlbam_id": "pitcher", "player_name": "pitcher_name"}),
        on="pitcher",
        how="left",
    )
)

missing_batter_names = named["batter_name"].isna().sum()
missing_pitcher_names = named["pitcher_name"].isna().sum()

print(f"Joined rows: {len(named):,}")
print(f"Missing batter names: {missing_batter_names:,}")
print(f"Missing pitcher names: {missing_pitcher_names:,}")

lookup_cols = [
    "game_date",
    "batter",
    "batter_name",
    "pitcher",
    "pitcher_name",
    "pitch_type",
    "description",
    "events",
]
display(named[lookup_cols].head(10))

print("\nLook up one batter ID:")
example_batter = int(named["batter"].iloc[0])
print(f"  ID {example_batter} → {players.loc[players['mlbam_id'] == example_batter, 'player_name'].iloc[0]}")

print("\nLook up one pitcher ID:")
example_pitcher = int(named["pitcher"].iloc[0])
print(f"  ID {example_pitcher} → {players.loc[players['mlbam_id'] == example_pitcher, 'player_name'].iloc[0]}")

Joined rows: 711,897
Missing batter names: 0
Missing pitcher names: 0


,game_date,batter,batter_name,pitcher,pitcher_name,pitch_type,description,events
0,2025-03-27,595777,Jurickson Profar,650633,Mike King,FF,called_strike,<NA>
1,2025-03-27,595777,Jurickson Profar,650633,Mike King,SI,ball,<NA>
2,2025-03-27,595777,Jurickson Profar,650633,Mike King,CH,hit_into_play,single
3,2025-03-27,663586,Austin Riley,650633,Mike King,FF,swinging_strike,<NA>
4,2025-03-27,663586,Austin Riley,650633,Mike King,FF,ball,<NA>
5,2025-03-27,663586,Austin Riley,650633,Mike King,SI,ball,<NA>
6,2025-03-27,663586,Austin Riley,650633,Mike King,CH,hit_into_play,field_out
7,2025-03-27,621566,Matt Olson,650633,Mike King,SI,called_strike,<NA>
8,2025-03-27,621566,Matt Olson,650633,Mike King,ST,ball,<NA>
9,2025-03-27,621566,Matt Olson,650633,Mike King,CH,ball,<NA>



Look up one batter ID:
  ID 595777 → Jurickson Profar

Look up one pitcher ID:
  ID 650633 → Mike King
